<h1 align="center"> Aplicações em Processamento de Linguagem Natural </h1>
<h2 align="center"> Aula 05 - Vetores de Palavras</h2>
<h3 align="center"> Prof. Fernando Vieira da Silva MSc.</h3>

<h2>1. Introdução</h2>

Vetores de palavras, ou Word2Vec, são representações muito usadas recentemente. Essas técnicas consistem em, geralmente, utilizar redes neurais artificiais simples com o objetivo de, dado uma palavra, prever a probabilidade da ocorrência de outras palavras nas proximidades. Porém, não se usa a rede neural treinada, mas os valores aprendidos como pesos da camada escondida são utilizados como representação para a palavra dada. Vamos estudar algumas arquiteturas diferentes.</p>


<h2>2. Arquitetura Skip-Gram </h2>
<p> A Figura 1 abaixo, retirada de (McCormick, 2016), mostra como são obtidas as amostras para treinar uma rede neural no modelo Word2Vec com arquitetura Skip-Gram para a frase “The quick brown fox jumps over the lazy dog” (“A rápida raposa marrom pula sobre o cachorro preguiçoso”, numa tradução livre). Nesse exemplo, a palavra usada como entrada da rede neural é marcada em azul, enquanto o resultado esperado é uma das palavras dentro do quadro em destaque (que trata-se da janela escolhida para delinear a proximidade). </p>
As amostras utilizadas são exibidas na coluna mais à direita.

![Figura 1: Amostras arquitetura Skip-gram](http://mccormickml.com/assets/word2vec/training_data.png)

O modelo de rede neural nessa arquitetura se assemelha ao exibido na Figura 3. Nesse outro exemplo, trata-se de um conjunto de 10 mil palavras e a entrada mostrada na figura é a palavra “ants” (formigas). Observe que esta palavra é representada em forma de um hot-vector, ou seja, um vetor com 10 mil dimensões (o mesmo número de palavras), onde somente uma leva o valor 1, enquanto as demais são iguais a 0. Há uma camada de neurônios escondida e uma camada de saída. Por sua vez, na camada de saída, cada neurônio retorna a probabilidade de uma palavra estar nas proximidades (também há 10 mil neurônios, ativados pela função Softmax). Essa rede é treinada com todas as ocorrências de cada palavra no corpus, e os pesos estabelecidos para a camada escondida são escolhidos como a representação dessa palavra (nesse exemplo, um vetor de 300 elementos).

![Figura 2: Arquitetura da rede neural Skip-Gram](http://mccormickml.com/assets/word2vec/skip_gram_net_arch.png)


<h2>3. Arquitetura CBOW (Continuous Bag of Words)</h2>

<p>Já na arquitetura CBOW, a entrada é a representação vetorial das palavras que cercam a palavra que se deseja representar, e a saída é justamente tal palavra, como ilustrado na figura abaixo.
    
![](https://cdn-images-1.medium.com/max/800/1*UVe8b6CWYykcxbBOR6uCfg.png)
<p fontsize="small">Fonte: https://arxiv.org/pdf/1301.3781.pdf Mikolov el al.</p>

Em resumo, enquanto o modelo na arquitetura Skip-Gram tenta prever o contexto, dado a palavra; a arquitetura CBOW tenta prever a palavra, dado o contexto.

Mikolov (autor) fez a seguinte comparação entre as duas arquiteturas:
* Skip-gram: Funciona bem com poucos dados de treino, representa bem até mesmo palavras ou frases raras.
* CBOW: muitas vezes mais rápido para treinar do qeu o Skip-gram, acurácia levemente melhor para palavras mais frequentes.

<p>Este artigo traz um exemplo bem interessante em que um modelo de rede neural na arquitetura Skip-Gram é criado usando o Keras: https://www.kdnuggets.com/2018/04/implementing-deep-learning-methods-feature-engineering-text-data-skip-gram.html, porém, por simplicidade, vamos treinar usando a biblioteca Gensim, que já possui os modelos adequados para Skip-Gram e CBOW.</p>


<p>Vamos gerar representações Word2Vec do livro Hamlet. Primeiro, vamos tokenizar o corpus, como já fizemos em aulas anteriores.</p>

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string

hamlet_raw = nltk.corpus.gutenberg.raw('shakespeare-hamlet.txt')
sentences = sent_tokenize(hamlet_raw)

all_words = [nltk.word_tokenize(sent) for sent in sentences]

for i in range(len(all_words)):  
    all_words[i] = [w.lower() for w in all_words[i] if w not in stopwords.words('english') and w not in string.punctuation]
    
print(all_words[:10])

In [ ]:
from gensim.models import Word2Vec

#sg ({0, 1}, optional) – Training algorithm: 1 for skip-gram; otherwise CBOW.
word2vec = Word2Vec(all_words, min_count=2, sg=1)

vocabulary = word2vec.wv.vocab  
print(vocabulary)

In [ ]:
v1 = word2vec.wv['heart']
print(v1)

In [ ]:
sim_words = word2vec.wv.most_similar('truth')
print(sim_words)

<h2>4. Arquitetura GloVe (Global Vectors)</h2>

<p>GloVe é um método de aprendizado não supervisionado que gera representações vetoriais de palavras com base na probabilidade de co-ocorrência entre duas palavras. Ela toma como base o vetor que representa a distância entre a representação vetorial dessas palavras, como pode ser observado na imagem abaixo. </p>

![GloVe - Distância vetorial ](https://nlp.stanford.edu/projects/glove/images/man_woman.jpg)

 <p>A intuição por trás do GloVe está no fato de que a probabilidade de co-ocorrência entre duas palavras será sempre alta para palavras de mesmo contexto (que são naturalmente associadas), e baixa para palavras sem relação de contexto. Esses dois casos representam ruído no modelo, como pode ser observado no exemplo abaixo, extraído de um corpus real de 2 bi. de palavras.</p>

![GloVe - Intuição sobre a co-ocorrência das palavras](https://nlp.stanford.edu/projects/glove/images/table.png)

<p>O objetivo do GloVe é aprender uma representação vetorial de palavras tal que o produto escalar entre duas palavras seja igual ao log da probabilidade de co-ocorrência. Isso é feito através de um modelo simples usando o algoritmo de mínimos quadrados. Dessa forma, é possível representar o significado das palavras.</p>

Vamos ver um exemplo prático de representação utilizando o glove python, usando um código inspirado em https://medium.com/@japneet121/word-vectorization-using-glove-76919685ee0b


In [ ]:
!pip install glove_python

In [ ]:
from glove import Corpus, Glove


corpus = Corpus() 

corpus.fit(all_words, window=10)

#creating a Glove object which will use the matrix created in the above lines to create embeddings
#We can set the learning rate as it uses Gradient Descent and number of components

glove = Glove(no_components=5, learning_rate=0.05)
 
glove.fit(corpus.matrix, epochs=30, no_threads=1, verbose=True)
glove.add_dictionary(corpus.dictionary)
glove.save('glove.model')

glove.add_dictionary(corpus.dictionary)

In [ ]:
print(corpus.dictionary)

In [ ]:
print(corpus.matrix)

In [ ]:
# Vamos ver os vetores de uma palavra:
print(glove.word_vectors[glove.dictionary['heart']])


In [ ]:
glove.most_similar('truth')

<b>Referências:</b>

<p>https://stackabuse.com/implementing-word2vec-with-gensim-library-in-python/</p>
<p>McCormick, C. (2016, April 19). Word2Vec Tutorial - The Skip-Gram Model. Retrieved from http://www.mccormickml.com</p>
<p>Jeffrey Pennington, Richard Socher, and Christopher D. Manning. 2014. GloVe: Global Vectors for Word Representation. </p>
<p>https://medium.com/@japneet121/word-vectorization-using-glove-76919685ee0b</p>
<p>Mikolov, Tomas & Corrado, G.s & Chen, Kai & Dean, Jeffrey. (2013). Efficient Estimation of Word Representations in Vector Space. 1-12. </p>

<p><b>Exercício 5:</b>Treine novamente seu modelo de identificação de sentimentos (positivo e negativo) usando o corpus disponível no NLTK (ver enunciado Exercício 4). Dessa vez, porém, use as representações Skip-Gram, CBOW e GloVe. Qual modelo apresenta melhor resultado? Como os resultados se comparam aos resultados obtidos com o modelo LSA usado anteriormente?
</p>

Dica: Segue abaixo código de um fit-transformer para Word2Vec

In [ ]:
from gensim.models import Word2Vec

#sg ({0, 1}, optional) – Training algorithm: 1 for skip-gram; otherwise CBOW.


class Word2VecTransformer(object):
    
    ALGO_SKIP_GRAM=1
    ALGO_CBOW=2    
    
    def __init__(self, algo=1):
        self.algo = algo
    
    def fit(self, X, y=None):     
        X = [nltk.word_tokenize(x) for x in X]
        
        self.word2vec = Word2Vec(X, min_count=2, sg=self.algo)
        
        # Pegamos a dimensão da primeira palavra, para saber quantas dimensões estamos trabalhando,
        # assim podemos ajustar nos casos em que aparecerem palavras que não existirem no vocabulário.
        first_word = next(iter(self.word2vec.wv.vocab.keys()))
        self.num_dim = len(self.word2vec[first_word])       
        
        return self
    
    def transform(self, X, Y=None):        
        X = [nltk.word_tokenize(x) for x in X]
        
        return np.array([np.mean([self.word2vec[w] for w in words if w in self.word2vec] or [np.zeros(self.num_dim)], axis=0) 
                         for words in X])

        
    def get_params(self, deep=True):
        return {}
